# P5-3. 최종 미니프로젝트 — 머신러닝 모델링 — 코드 구현 가이드

**주제: 고객 이탈(Churn) 예측 — 머신러닝 기준 모델 만들기**

## 수업 목적
P5 과정에서 학습한 머신러닝 내용을 바탕으로, 완성 코드를 그대로 따라 입력하지 않고
**데이터 준비 → 전처리 → 모델 학습 → 불균형 대응 → 평가 → 해석**의 전체 흐름을 직접 구현한다.

## 핵심 구현 흐름
`데이터 로드 → 데이터 확인 → One-Hot Encoding → X/y 분리 → Train/Test 분할 → 스케일링 → 모델 학습 → class_weight → 성능 비교 → 상세 평가`

## 구현 원칙
- `Churn`은 정답 데이터이므로 입력 특성에 포함하지 않는다.
- P5-4와 공정하게 비교할 수 있도록 `test_size=0.30`, `random_state=42`를 사용한다.
- 클래스 비율을 유지하도록 `stratify=y`를 적용한다.
- 스케일러는 Train 데이터에만 `fit`한다.
- Accuracy뿐 아니라 Precision, Recall, F1-score를 함께 비교한다.
- Test 데이터는 최종 평가용으로 사용하고 반복적인 모델 튜닝에 사용하지 않는다.

## 0. 라이브러리 준비
필요한 `pandas`, `numpy`, 시각화 도구와 scikit-learn의 전처리·분류·평가 모듈을 직접 불러온다.

**힌트:** `train_test_split`, `MinMaxScaler`, 4개의 분류 모델, 분류 평가지표가 필요하다.

In [ ]:
# 필요한 라이브러리를 직접 작성한다.

## 1. 데이터 로드
`data_save.csv`를 DataFrame으로 읽고 데이터 크기와 앞부분을 확인한다.

**확인할 것:** 행/열 개수, `Churn` 컬럼 존재 여부

In [ ]:
# data_save.csv를 불러오고 데이터 크기와 앞 5개 행을 확인한다.

## 2. 데이터 확인
`df.info()`로 자료형과 결측치를 확인하고 `Churn`의 개수 및 비율을 계산한다.

**생각할 것:** `Churn=1`이 적다면 Accuracy만으로 충분한가?

In [ ]:
# 데이터 구조와 Churn 클래스 분포를 확인한다.

## 3. 범주형 데이터 One-Hot Encoding
문자열형 컬럼을 찾아 `pd.get_dummies()`로 수치형으로 변환한다.

**힌트:** `select_dtypes()`로 범주형 컬럼을 찾을 수 있다.

In [ ]:
# 범주형 컬럼을 찾고 One-Hot Encoding을 수행한다.

## 4. 입력(X)과 정답(y) 분리
`Churn`을 제외한 컬럼은 X, `Churn`은 y로 분리한다.

In [ ]:
# X와 y를 분리하고 shape을 확인한다.

## 5. Train/Test 분할
P5-4와 같은 Test 데이터를 만들 수 있도록 다음 조건을 적용한다.

- Test 비율: 30%
- `random_state=42`
- 클래스 비율 유지: `stratify=y`

In [ ]:
# train_test_split()으로 Train/Test를 분리한다.

## 6. 스케일링
`MinMaxScaler`를 사용한다.

**중요**
1. Train 데이터로 `fit_transform()`
2. Test 데이터에는 같은 scaler로 `transform()`만 적용한다.

이 순서를 지켜야 데이터 누수를 방지할 수 있다.

In [ ]:
# MinMaxScaler를 만들고 Train/Test 데이터를 변환한다.

## 7. 공통 평가 함수 구현
모든 모델을 동일한 기준으로 비교할 수 있도록 평가 함수를 만든다.

함수는 다음 값을 계산하도록 한다.
- Accuracy
- Precision
- Recall
- F1-score

**힌트:** 함수 입력은 `모델 이름, 학습된 모델, 평가 입력 데이터, 실제 정답` 정도로 구성할 수 있다.

In [ ]:
# evaluate_model() 함수를 직접 구현한다.

## 8. 머신러닝 기본 모델 학습
다음 4개 모델을 같은 Train/Test 데이터로 학습하고 평가한다.

- Logistic Regression
- KNN
- Decision Tree
- Random Forest

**권장 설정**
- Logistic Regression: `max_iter=1000`
- KNN: `n_neighbors=5`
- Decision Tree: `max_depth=10`
- Random Forest: `n_estimators=100`

각 모델의 결과와 예측값을 저장한다.

In [ ]:
# 4개 모델을 정의하고 반복문으로 학습·평가한다.

## 9. 클래스 불균형 대응 — class_weight
다음 두 모델에 `class_weight="balanced"`를 적용한 모델을 추가한다.

- Logistic Regression
- Random Forest

기본 모델과 Recall, Precision, F1의 변화가 어떻게 나타나는지 비교한다.

In [ ]:
# class_weight='balanced' 모델을 추가 학습하고 결과를 저장한다.

## 10. 최종 성능 비교
모든 결과를 하나의 DataFrame으로 정리한다.

**권장 정렬 기준**
1. Recall 내림차순
2. F1 내림차순

Recall과 F1을 막대그래프로 비교해도 좋다.

In [ ]:
# 결과 DataFrame을 만들고 성능을 비교한다.

## 11. 선택 모델 상세 평가
프로젝트 목적을 **'이탈 고객을 놓치지 않는 것'**으로 가정한다.

Recall이 높은 대표 모델 하나를 선택하여 다음을 출력한다.
- `classification_report`
- Confusion Matrix

**해석 포인트:** False Negative가 어떤 의미인지 설명한다.

In [ ]:
# 대표 모델을 선택하고 상세 평가를 수행한다.

## 12. 결과 해석
다음 질문에 답한다.

1. Accuracy가 가장 높은 모델과 Recall이 가장 높은 모델이 같은가?
2. `class_weight='balanced'` 적용 전후 Recall과 Precision은 어떻게 달라졌는가?
3. 이탈 고객을 놓치는 비용이 크다면 어떤 모델을 선택하겠는가?
4. P5-4의 DNN과 비교할 머신러닝 대표 모델은 무엇인가?

## 최종 점검표
| 확인 항목 | 핵심 기준 |
|---|---|
| 전처리 | 범주형 컬럼을 One-Hot Encoding 했는가 |
| 데이터 분리 | P5-4와 동일한 Test 분할 조건을 사용했는가 |
| Stratify | 클래스 비율을 유지했는가 |
| 스케일링 | Train에만 fit했는가 |
| 모델 | 4개 기본 모델을 비교했는가 |
| 불균형 대응 | class_weight 적용 모델을 추가했는가 |
| 평가 | Accuracy, Precision, Recall, F1을 사용했는가 |
| 상세 분석 | classification report와 confusion matrix를 확인했는가 |
| 해석 | 성능 숫자뿐 아니라 프로젝트 목적에 맞게 모델을 선택했는가 |